# Grounded Learning RAG System

A compact notebook structure for PDF ingestion, retrieval, grounded QA, summaries, quizzes, and flashcards.

## 1. Environment Setup

Install dependencies and download the required data files.

In [ ]:
!pip uninstall -y torchcodec

!gdown 140Gss4XS0Mrqtog9bNNWzHfdKuoIKDvc -O requirements.txt

!mkdir -p data
!gdown 1yTGI5xOk0fi6BsQwACbhmBTrnLr31DJe -O data/data.zip
!unzip -o data/data.zip -d data

!pip install -q -r requirements.txt pypdf


## 2. Core Imports

Load the standard libraries and validation utilities used across the notebook.

In [ ]:
from pathlib import Path
from loguru import logger
import json
import re

from pydantic import BaseModel, Field
from typing import Literal, List, Optional
from pydantic_settings import BaseSettings


## 3. Data Schemas

Define typed objects for retrieved chunks, citations, summaries, quizzes, and flashcards.

In [ ]:
class ChunkMetadata(BaseModel):
    document_id: str
    filename: str
    source: str
    page: int
    chunk_id: str
    section: Optional[str] = None

class RetrievedChunk(BaseModel):
    text: str
    score: float
    metadata: ChunkMetadata

class Citation(BaseModel):
    source_index: int
    source_marker: str
    filename: str
    page: int
    section: Optional[str] = None
    chunk_id: str

class Summary(BaseModel):
    scope: str
    target: Optional[str] = None
    summary: str
    key_points: List[str]
    citations: List[Citation]

class QuizItem(BaseModel):
    question: str
    options: List[str]
    correct_index: int
    explanation: str
    source_markers: List[str]
    difficulty: Optional[str] = None
    topic: Optional[str] = None

class QuizSet(BaseModel):
    scope: str
    target: Optional[str] = None
    items: List[QuizItem]
    citations: List[Citation]

class Flashcard(BaseModel):
    front: str
    back: str
    hint: Optional[str] = None
    topic: Optional[str] = None
    source_markers: List[str]

class FlashcardSet(BaseModel):
    scope: str
    target: Optional[str] = None
    cards: List[Flashcard]
    citations: List[Citation]


## 4. System Configuration

Set paths, embedding model, LLM provider, and generation defaults.

In [ ]:
class Settings(BaseSettings):
    data_dir: Path = Path("data")
    storage_dir: Path = Path("storage/qdrant")
    qdrant_collection: str = "rag_chunks" #collection_name

    chunk_size: int = Field(default=1000, ge=100)
    chunk_overlap: int = Field(default=150, ge=0)
    top_k: int = Field(default=5, ge=1, le=64)

    embedding_model: str = "keepitreal/vietnamese-sbert"

    llm_provider: Literal["hf_local", "gemini", "vllm"] = "hf_local"
    llm_temperature: float = Field(default=0.1, ge=0.0, le=2.0)

    hf_model: str = "Qwen/Qwen2.5-1.5B-Instruct"
    hf_device: int = 0
    hf_max_new_tokens: int = Field(default=1024, ge=1)

    gemini_model: str = "gemini-2.5-flash"

    summarize_batch_size: int = Field(default=10, ge=1)
    summarize_retrieval_k: int = Field(default=12, ge=1, le=128)
    generation_retrieval_k: int = Field(default=16, ge=1, le=128)
    quiz_default_count: int = Field(default=5, ge=1, le=50)
    flashcards_default_count: int = Field(default=5, ge=1, le=100)

settings = Settings()
settings.data_dir.mkdir(parents=True, exist_ok=True)
settings.storage_dir.mkdir(parents=True, exist_ok=True)


## 5. RAG Dependencies

Import libraries for PDF loading, chunking, embeddings, and Qdrant storage.

In [ ]:
import hashlib
import uuid
from collections import defaultdict

from langchain_community.document_loaders.pdf import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient
from qdrant_client.http import models as qmodels


## 6. Embedding Model

Create the embedding model used to convert text chunks into vectors.

In [ ]:
def get_embeddings():
    return HuggingFaceEmbeddings(
        model_name= # YOUR CODE HERE
        model_kwargs={"device": "cuda"},
        encode_kwargs={"normalize_embeddings": True},
    )


## 7. Vector Store Setup

Create or reuse the local Qdrant collection for storing document vectors.

In [ ]:
def get_client() -> QdrantClient:
    return QdrantClient(path=str(settings.storage_dir))


# Use QdrantClient's methods to check if a collection exists or not.

def ensure_collection(recreate: bool = False):
    client = get_client()
    if recreate and client.collection_exists(settings.qdrant_collection):
        # YOUR CODE HERE

    if not client.collection_exists(settings.qdrant_collection):
        dim = len(get_embeddings().embed_query("test"))
        # YOUR CODE HERE


def get_vector_store():
    return QdrantVectorStore(
        client= # YOUR CODE HERE
        collection_name= # YOUR CODE HERE
        embedding= # YOUR CODE HERE
    )


## 8. PDF Chunk Builder

Load PDF pages, attach metadata, and split text into retrievable chunks.

In [ ]:
def build_chunks(pdf_paths):
    page_docs = []

    for path in pdf_paths:
        loader = PyPDFLoader(str(path))
        pages = loader.load()

        doc_id = hashlib.sha1(f"{path.name}:{path.stat().st_size}".encode("utf-8")).hexdigest()[:16]

        for doc in pages:
            doc.metadata = {
                "document_id": doc_id,
                "filename": path.name,
                "source": str(path.resolve()),
                "page": int(doc.metadata.get("page", 0)) + 1,
            }
        page_docs.extend(pages)

    splitter = RecursiveCharacterTextSplitter(
        chunk_size= # YOUR CODE HERE,
        chunk_overlap= # YOUR CODE HERE,
        separators= # YOUR CODE HERE,
    )
    chunks = splitter.split_documents(page_docs)

    per_doc_counter = defaultdict(int)
    for chunk in chunks:
        doc_id = chunk.metadata["document_id"]
        idx = per_doc_counter[doc_id]
        per_doc_counter[doc_id] += 1
        chunk.metadata["chunk_id"] = f"{doc_id}:{chunk.metadata['page']}:{idx}"

    return chunks


## 9. Single PDF Ingestion

Ingest one PDF file into the vector store.

In [ ]:
def save_and_ingest_pdf(file_path: str):
    path = Path(file_path)
    ensure_collection(recreate=False)
    chunks = # YOUR CODE HERE
    if not chunks:
        return 0

    ids = [str(uuid.uuid5(uuid.NAMESPACE_DNS, c.metadata["chunk_id"])) for c in chunks]
    get_vector_store().add_documents(chunks, ids=ids)
    logger.info(f"Saved {len(chunks)} chunks from {path.name}")
    return len(chunks)


## 10. LLM Dependencies

Import model wrappers for local Hugging Face and Gemini backends.

In [ ]:
import torch
from langchain_huggingface import ChatHuggingFace, HuggingFacePipeline
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage


## 11. Local Hugging Face LLM

Build a local text-generation model for offline inference.

In [ ]:
def _build_hf_local():
    logger.info(f"Loading local model: {settings.hf_model}...")

    tokenizer = AutoTokenizer.from_pretrained(
        # YOUR CODE HERE
    )
    model = AutoModelForCausalLM.from_pretrained(
        # YOUR CODE HERE
    )

    text_gen_pipeline = pipeline(
        task="text-generation",
        model=model,
        tokenizer=tokenizer,
        max_new_tokens=settings.hf_max_new_tokens,
        do_sample=settings.llm_temperature > 0,
        temperature=settings.llm_temperature,
        return_full_text=False,
    )

    llm = # YOUR CODE HERE
    return ChatHuggingFace(llm=llm)


## 12. Gemini LLM

Build a Gemini chat model as an alternative generation backend.

In [ ]:
def _build_gemini():
    return ChatGoogleGenerativeAI(
        model= # YOUR CODE HERE
        temperature= # YOUR CODE HERE
    )


## 13. LLM Provider Selector

Select the configured model backend at runtime.

In [ ]:
def get_llm():
    provider = settings.llm_provider
    if provider == "hf_local":
        # YOUR CODE HERE
    elif provider == "gemini":
        # YOUR CODE HERE
    else:
        raise ValueError(f"Provider {provider} is not supported in this demo.")


## 14. Retrieval and Citations

Retrieve relevant chunks and convert them into citation metadata.

In [ ]:
def retrieve(query: str, k: int = 5):
    store = get_vector_store()
    hits = store.similarity_search_with_score(query=query, k=k)
    return [
        RetrievedChunk(
            text=# YOUR CODE HERE
            score=# YOUR CODE HERE
            metadata=ChunkMetadata(**doc.metadata),
        )
        for doc, score in hits
    ]


def format_citations(chunks):
    return [
        Citation(
            source_index=i,
            source_marker=f"S{i}",
            filename=c.metadata.filename,
            page=c.metadata.page,
            chunk_id=c.metadata.chunk_id,
        )
        for i, c in enumerate(chunks, start=1)
    ]


## 15. JSON Output Parser

Extract structured JSON from LLM responses.

In [ ]:
def _parse_json(text: str) -> dict:
    match = re.search(r'\{.*\}', text, re.DOTALL)

    if match:
        json_str = match.group(0)
        try:
            return json.loads(json_str)
        except json.JSONDecodeError as e:
            logger.error(f"Internal JSON parsing error: {e}")
            logger.error(f"Faulty string: {json_str}")
            return {}
    else:
        logger.error(f"No JSON structure found in LLM output: {text}")
        return {}


## 16. Answer Prompt Template

Build a grounded prompt for question answering with inline source markers.

In [ ]:
def build_answer_prompt(question: str, chunks: list) -> str:
    context_text = "\n".join([f"---\n[source=S{i+1}]\n{chunk.text}" for i, chunk in enumerate(chunks)])

    prompt = f"""You are a precise assistant. Answer the user's question using ONLY the context below.

Rules:
- Use only facts explicitly supported by the context. Do not invent details.
- If the context is insufficient, reply exactly: "I do not have enough information in the provided context to answer."
- Be concise and direct.
- Write your answer in English.
- Cite support inline using source markers like [S1], [S2].
- Use only the source markers provided in the context.
- Do not write filenames, page numbers, or chunk IDs in the answer body.

Context:
{context_text}

Question: {question}

Answer:"""
    return prompt


## 17. Flashcard Prompt Template

Build a grounded prompt for study flashcard generation.

In [ ]:
def build_flashcards_prompt(chunks: list, count: int) -> str:
    context_text = "\n".join([
        f"---\n[source=S{i+1}] ({chunk.metadata.filename} p.{chunk.metadata.page})\n{chunk.text}"
        for i, chunk in enumerate(chunks)
    ])

    prompt = f"""You are generating study flashcards grounded in the provided context.

Rules:
- Generate exactly {count} flashcards.
- Fronts should ask for a concept, term, distinction, formula, process, or short explanation.
- Backs must be concise but complete enough for revision.
- Ground every card in the context. Do not invent facts.
- Avoid low-value cards that only restate section headings.
- Avoid duplicate or overly similar cards.
- Provide at least one [S#] source marker per card when possible.
- If the context is insufficient for {count} useful cards, produce fewer rather than pad.
- Write all fronts, backs, hints, and topic labels in English.

Output STRICTLY valid JSON and no other text, with this exact shape:
{{
  "cards": [
    {{
      "front": "string",
      "back": "string",
      "hint": "optional string or null",
      "topic": "optional short topic label or null",
      "source_markers": ["S1"]
    }}
  ]
}}

Context:
{context_text}

Respond with ONLY the JSON object."""
    return prompt


## 18. Quiz Prompt Template

Build a grounded prompt for multiple-choice quiz generation.

In [ ]:
def build_quiz_prompt(chunks: list, count: int) -> str:
    context_text = "\n".join([
        f"---\n[source=S{i+1}] ({chunk.metadata.filename} p.{chunk.metadata.page})\n{chunk.text}"
        for i, chunk in enumerate(chunks)
    ])

    prompt = f"""You are generating high-quality multiple-choice quiz items grounded in the provided context.

Rules:
- Generate exactly {count} items.
- Every question must be answerable from the context alone.
- Provide exactly 4 options per question with exactly one correct answer.
- Test understanding: concepts, distinctions, reasoning, and factual recall.
- Avoid duplicates or near-duplicates.
- Avoid trick questions or ambiguous wording.
- Explanations must be concise and cite at least one [S#] marker from the context.
- If the context is insufficient to create {count} high-quality items, generate fewer. Never fabricate facts.
- Write all questions, options, explanations, and topic labels in English.

Output STRICTLY valid JSON and no other text, with this exact shape:
{{
  "items": [
    {{
      "question": "string",
      "options": ["string", "string", "string", "string"],
      "correct_index": 0,
      "explanation": "grounded explanation with [S1] style citations",
      "source_markers": ["S1"],
      "difficulty": "easy|medium|hard",
      "topic": "short topic label"
    }}
  ]
}}

Context:
{context_text}

Respond with ONLY the JSON object."""
    return prompt


## 19. Summary Prompt Template

Build a grounded prompt for study-oriented summarization.

In [ ]:
def build_summary_single_prompt(chunks: list) -> str:
    context_text = "\n".join([
        f"---\n[source=S{i+1}] ({chunk.metadata.filename} p.{chunk.metadata.page})\n{chunk.text}"
        for i, chunk in enumerate(chunks)
    ])

    prompt = f"""You are writing a study-oriented summary grounded strictly in the provided context.

Rules:
- Use only facts explicitly supported by the context. Do not invent details.
- Do not add outside knowledge.
- Focus on concepts, definitions, relationships, and reasoning a learner should retain.
- Keep the tone clear, neutral, and practical for study.
- Write the summary and key points in English.
- If the context is empty or unrelated, return an empty summary and an empty list of key points.

Output STRICTLY valid JSON with this shape and no extra text:
{{
  "summary": "A coherent multi-paragraph study summary.",
  "key_points": ["concise learnable fact", "concise learnable fact"]
}}

Context:
{context_text}

Respond with ONLY the JSON object."""
    return prompt


## 20. Question Answering Feature

Retrieve relevant context and answer a user question with citations.

In [ ]:
def answer(question: str, k: int = 5):
    chunks = # YOUR CODE HERE
    if not chunks:
        return {"answer": "I could not find information in the documents.", "citations": []}

    prompt = # YOUR CODE HERE
    response = # YOUR CODE HERE (use invoke method))
    return {
        "answer": response.content,
        "citations": format_citations(chunks),
    }


## 21. Flashcard Generation Feature

Generate study flashcards from retrieved document context.

In [ ]:
def generate_flashcards(query: str, count: int = 3):
    chunks = # YOUR CODE HERE
    prompt = # YOUR CODE HERE

    response = # YOUR CODE HERE
    payload = _parse_json(response.content)

    cards = [Flashcard(**card) for card in payload.get("cards", [])]
    return FlashcardSet(scope="query", target=query, cards=cards, citations=format_citations(chunks))


## 22. Summary Generation Feature

Generate a study-oriented summary from retrieved document context.

In [ ]:
def summarize_fixed(query: str):
    chunks = # YOUR CODE HERE
    prompt = # YOUR CODE HERE

    response = # YOUR CODE HERE
    payload = _parse_json(response.content)

    return Summary(
        scope="query",
        target=query,
        summary=payload.get("summary", ""),
        key_points=payload.get("key_points", []),
        citations=format_citations(chunks),
    )


## 23. Quiz Generation Feature

Generate multiple-choice questions from retrieved document context.

In [ ]:
def generate_quiz(query: str, count: int = 3):
    chunks = # YOUR CODE HERE
    prompt = # YOUR CODE HERE

    response = # YOUR CODE HERE
    payload = _parse_json(response.content)

    items = [QuizItem(**item) for item in payload.get("items", [])]
    return QuizSet(
        scope="query",
        target=query,
        items=items,
        citations=format_citations(chunks),
    )


## 24. Batch PDF Ingestion

Find all PDFs in the data folder and ingest them into Qdrant.

In [ ]:
def ingest_data_directory():
    """Automatically find and load all PDF files in the configured data directory."""
    ensure_collection(recreate=False)

    pdf_paths = list(settings.data_dir.glob("*.pdf"))

    if not pdf_paths:
        logger.warning(f"No PDF files found in directory: {settings.data_dir}")
        return 0

    logger.info(f"Found {len(pdf_paths)} PDF files. Proceeding with chunking...")

    chunks = build_chunks(pdf_paths)

    if not chunks:
        return 0

    ids = [str(uuid.uuid5(uuid.NAMESPACE_DNS, c.metadata["chunk_id"])) for c in chunks]
    get_vector_store().add_documents(chunks, ids=ids)

    logger.info(f"Successfully ingested {len(chunks)} text chunks from {len(pdf_paths)} documents.")
    return len(chunks)


## 25. End-to-End Demo

Run ingestion and test all four learning features.

In [ ]:
import urllib.request

settings.data_dir.mkdir(parents=True, exist_ok=True)
logger.info("STEP 1: CHECKING & PREPARING DATA AT /content/data")

if not any(settings.data_dir.iterdir()):
    logger.info("Directory is empty, downloading sample PDF for testing...")
    sample_url = "https://www.w3.org/WAI/ER/tests/xhtml/testfiles/resources/pdf/dummy.pdf"
    urllib.request.urlretrieve(sample_url, settings.data_dir / "sample_dummy.pdf")
else:
    logger.info(f"Found documents in {settings.data_dir}. Skipping sample download.")

logger.info("STEP 2: INGESTING DIRECTORY INTO VECTOR STORE (QDRANT)")
total_chunks = ingest_data_directory()
print(f"Successfully chunked and saved {total_chunks} text segments into Qdrant.\n")

if total_chunks > 0:
    logger.info("STEP 3: RUNNING 4 LEARNING FEATURES")

    print("\n" + "=" * 55)
    print("FEATURE 1: QUESTION ANSWERING WITH CITATIONS (ANSWER)")
    print("=" * 55)
    question = "What is the main content of this document?"
    ans_res = answer(question)
    print(f"Question: {question}")
    print(f"Answer: {ans_res['answer']}")
    print("\nCitations:")
    for c in ans_res["citations"]:
        print(f"   - File: {c.filename} | Page: {c.page} | Marker: [{c.source_marker}]")

    print("\n" + "=" * 55)
    print("FEATURE 2: DOCUMENT SUMMARY (SUMMARY)")
    print("=" * 55)
    sum_res = summarize_fixed("Summarize the most important points of the document.")
    print(f"Overall Summary:\n{sum_res.summary}\n")
    print("Key Points:")
    for kp in sum_res.key_points:
        print(f"  * {kp}")

    print("\n" + "=" * 55)
    print("FEATURE 3: GENERATE MULTIPLE CHOICE QUIZ (QUIZ)")
    print("=" * 55)
    quiz_res = generate_quiz("Create a quiz to test knowledge", count=2)
    for idx, q in enumerate(quiz_res.items, 1):
        print(f"Question {idx}: {q.question}")
        for opt in q.options:
            print(f"  {opt}")
        print(f"=> Explanation: {q.explanation} (Difficulty: {q.difficulty})")
        print("-" * 30)

    print("\n" + "=" * 55)
    print("FEATURE 4: GENERATE FLASHCARDS (FLASHCARDS)")
    print("=" * 55)
    flash_res = generate_flashcards("Create vocabulary or concept flashcards", count=3)
    for idx, card in enumerate(flash_res.cards, 1):
        print(f"Card {idx}:")
        print(f"  Front (Question): {card.front}")
        print(f"  Back (Answer)   : {card.back}")
        if card.hint:
            print(f"  Hint            : {card.hint}")
        print("-" * 30)

    logger.info("ALL FEATURES EXECUTED SUCCESSFULLY!")
